In [1]:
import pandas as pd
import numpy as np
import os
import glob

In [2]:

season = 2627
# 1. LOAD AND COMBINE ALL GW FILES
# -----------------------------------------------
# Finds every CSV in your GW folder and stacks them into one table

folder = r"C:\Users\JesseOnu\fpl sql rework\gws"
#all_files = glob(os.path.join(folder, "*.csv"))
# Only get files that START with the season and end with .csv
all_files = glob.glob(os.path.join(folder, f"{season}*.csv"))

# Optional: sort the files (recommended)
all_files = sorted(all_files)

print(f"Found {len(all_files)} files for season {season}:")
#for file in all_files:
    #print("   ", os.path.basename(file))

Found 2 files for season 2627:


In [3]:
#all_files = glob(os.path.join(folder, "*.csv"))
df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

# -----------------------------------------------
# 2. FILTER TO PLAYERS WITH MINUTES > 0
# -----------------------------------------------
# Removes players who never played - stops them distorting percentiles
df = df[df["minutes"] > 0].copy()

# -----------------------------------------------
# 3. MINUTES SCORE (max-based, not percentile)
# -----------------------------------------------
# Max possible mins = highest gameweek number * 90
# Each player gets a % of that, scaled to 1-10
max_gw = df["gameweek"].max()
max_possible_mins = max_gw * 90

# Sum each player's total minutes across all GWs up to each GW
# We calculate cumulative minutes per player per GW
df = df.sort_values(["player_code", "gameweek"])
df["cumulative_mins"] = df.groupby("player_code")["minutes"].cumsum()
df["mins_score"] = (df["cumulative_mins"] / max_possible_mins * 9 + 1).clip(1, 10)

# -----------------------------------------------
# 4. PERCENTILE SCORES (within position, cumulative)
# -----------------------------------------------
# For each metric, we:
#   a) Calculate each player's cumulative total up to that GW
#   b) Rank them within their position at that GW snapshot
#   c) Convert rank to a 1-10 score

# Metrics where MORE = BETTER
positive_metrics = {
    "goals_scored": "goals_score",
    "assists":      "assists_score",
    "xG":           "xg_score",
    "xGI":          "xgi_score",
    "clean_sheets": "cs_score",
    "bonus":        "bonus_score",
}

# Metrics where LESS = BETTER (we invert the percentile)
negative_metrics = {
    "goals_conceded": "goals_conceded_score",
    "xGC":            "xgc_score",
}

def percentile_score(series, invert=False):
    """
    Takes a series of cumulative values.
    Returns a 1-10 score based on percentile rank.
    If invert=True, lower values get higher scores.
    """
    pct = series.rank(pct=True)   # gives each value a 0-1 percentile rank
    if invert:
        pct = 1 - pct             # flip so lower = better becomes higher score
    return (pct * 9 + 1).clip(1, 10)

# Calculate cumulative totals and percentile scores for each metric
for col, score_col in {**positive_metrics, **negative_metrics}.items():
    invert = col in negative_metrics
    
    # Cumulative total per player up to each GW
    cum_col = f"cum_{col}"
    df[cum_col] = df.groupby("player_code")[col].cumsum()
    
    # Percentile rank within position at each GW snapshot
    df[score_col] = (
        df.groupby(["gameweek", "position_id"])[cum_col]
        .transform(lambda x: percentile_score(x, invert=invert))
    )

# -----------------------------------------------
# 4.5. POSITION-ADJUST ATTACKING SCORES
# -----------------------------------------------
# xGI/xG/assists are weak or meaningless signals for GKs and less
# important for DEFs than MID/FWD. Rather than change the composite
# weights (section 5), we scale these score columns down by position
# BEFORE composite_score is calculated, so section 5 stays untouched.
#
# position_id convention assumed: 1=GK, 2=DEF, 3=MID, 4=FWD
# Multiplier of 1.0 = no change, lower = drastically reduced

position_multiplier = {
    "xgi_score":     {1: 0.1, 2: 0.4, 3: 1.0, 4: 1.0},
    "xg_score":      {1: 0.1, 2: 0.4, 3: 1.0, 4: 1.0},
    "assists_score": {1: 0.3, 2: 0.6, 3: 1.0, 4: 1.0},
}

def apply_position_adjustment(df, score_col, multiplier_dict):
    multiplier = df["position_id"].map(multiplier_dict)
    adjusted = df[score_col] * multiplier
    return adjusted.clip(1, 10)

for score_col, mult_dict in position_multiplier.items():
    df[score_col] = apply_position_adjustment(df, score_col, mult_dict)

# -----------------------------------------------
# 5. COMPOSITE SCORE (weighted average)
# -----------------------------------------------
# You can adjust these weights per position later
# For now, sensible defaults - same for all positions
# All weights should add up to 1
weights = {
    "mins_score":           0.1,
    "goals_score":          0.29,
    "assists_score":        0.10,
    "xg_score":             0.10,
    "xgi_score":            0.35,
    "cs_score":             0.10,
    "bonus_score":          0.05,
    "goals_conceded_score": 0.05,
    "xgc_score":            0.05,
}
df["composite_score"] = sum(
    df[score] * weight for score, weight in weights.items()
)

# -----------------------------------------------
# 6. SELECT OUTPUT COLUMNS
# -----------------------------------------------
# One row per player per GW with all scores
output_cols = [
    "player_code",
    "web_name",
    "fixture_id",
    "position_id",
    "gameweek",
    "minutes",
    "mins_score",
    "goals_score",
    "assists_score",
    "xg_score",
    "xgi_score",
    "cs_score",
    "bonus_score",
    "goals_conceded_score",
    "xgc_score",
    "composite_score",
]
output = df[output_cols].sort_values(["gameweek", "composite_score"], ascending=[True, False])
output['season'] = season

In [4]:
output["fixture_id"] = output["fixture_id"].astype("Int64")

In [5]:
output.head(5)

,player_code,web_name,fixture_id,position_id,gameweek,minutes,mins_score,goals_score,assists_score,xg_score,xgi_score,cs_score,bonus_score,goals_conceded_score,xgc_score,composite_score,season
122,532529,Hinshelwood,7,3,1,63,4.15,10.000000,5.102041,10.000000,10.000000,9.020408,9.785714,7.673469,7.612245,10.480816,2627
85,249231,Lewis-Potter,2,3,1,90,5.50,9.479592,5.102041,9.816327,9.816327,9.020408,9.448980,7.673469,5.897959,10.279694,2627
11,223340,Saka,1,3,1,67,4.35,9.479592,5.102041,9.755102,9.693878,9.020408,9.173469,7.673469,9.234694,10.268776,2627
97,204580,Janelt,2,3,1,90,5.50,9.479592,5.102041,9.387755,9.479592,9.020408,5.010204,7.673469,5.897959,9.897041,2627
14,184029,Ødegaard,1,3,1,75,4.75,9.479592,5.102041,8.806122,8.285714,9.020408,9.785714,7.673469,9.234694,9.751633,2627


In [6]:
# -----------------------------------------------
# 7. SAVE OUTPUT
# -----------------------------------------------
output_path = rf"C:\Users\JesseOnu\fpl sql rework\playerscores\{season}player_scores.csv"

output.to_csv(output_path, index=False)

print(f"Done! {len(output)} rows saved to {output_path}")
print(output.head(10))

Done! 622 rows saved to C:\Users\JesseOnu\fpl sql rework\playerscores\2627player_scores.csv
     player_code       web_name  fixture_id  position_id  gameweek  minutes  \
122       532529    Hinshelwood           7            3         1       63   
85        249231   Lewis-Potter           2            3         1       90   
11        223340           Saka           1            3         1       67   
97        204580         Janelt           2            3         1       90   
14        184029       Ødegaard           1            3         1       75   
153       244851         Palmer          10            3         1       82   
367       424876     Szoboszlai           9            3         1       90   
39        244850         Rogers          10            3         1       81   
235       215413  Dewsbury-Hall           3            3         1       90   
315       543968       Emersonn           5            4         1       65   

     mins_score  goals_score  assists_

In [7]:
df2 =  pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

In [8]:
df2.head()

,player_id,player_code,gameweek,fixture_id,name,web_name,position_id,selected_by_percent,value,team_id,...,influence,creativity,threat,in_dreamteam,status,news,corners_indirect_freekicks_order,direct_freekicks_order,penalties_order,season
0,1,154561,1,1,David Raya Martín,Raya,1,38.1,6.0,1,...,11.8,0.0,0.0,False,a,NaN,NaN,NaN,NaN,2627
1,2,109745,1,1,Kepa Arrizabalaga Revuelta,Arrizabalaga,1,0.1,5.0,1,...,0.0,0.0,0.0,False,a,NaN,NaN,NaN,NaN,2627
2,3,437495,1,1,Illan Meslier,Meslier,1,0.0,5.0,1,...,0.0,0.0,0.0,False,a,NaN,NaN,NaN,NaN,2627
3,4,226597,1,1,Gabriel dos Santos Magalhães,Gabriel,2,29.4,8.0,1,...,6.6,1.8,1.0,False,a,NaN,NaN,NaN,NaN,2627
4,5,445122,1,1,Jurriën Timber,J.Timber,2,0.1,6.5,1,...,0.0,0.0,0.0,False,i,Groin injury - Unknown return date,NaN,NaN,NaN,2627
